In [5]:
import os
import numpy as np
import torch
import torch.nn as nn
import pandas as pd
import cv2
import mediapipe as mp
import pandas as pd
import numpy as np
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from IPython.display import HTML
import joblib

if torch.backends.mps.is_available():
    device = torch.device("mps")      # Mac GPU (Apple Silicon)
elif torch.cuda.is_available():
    device = torch.device("cuda")     # Nvidia GPU
else:
    device = torch.device("cpu")

In [6]:
def convert_padded_csv_to_fixed_c(
    input_folder,
    output_folder,
    C=30):
    
    os.makedirs(output_folder, exist_ok=True)

    for file_name in os.listdir(input_folder):

        if not file_name.endswith(".csv"):
            continue

        print(f"\nProcessing: {file_name}")

        csv_path = os.path.join(input_folder, file_name)

        df = pd.read_csv(csv_path)

        target = df["target"].iloc[0]

        X_flat = df.drop(columns=["target"]).values

        # reshape back to frames
        X = X_flat.reshape(-1, 39)

        # remove zero rows
        non_zero_mask = ~(np.all(X == 0, axis=1))
        X_trimmed = X[non_zero_mask]

        print(f"Original frames: {len(X)}")
        print(f"After zero removal: {len(X_trimmed)}")

        if len(X_trimmed) < C:
            print(f"Skipping {file_name}: too short")
            continue

        # fixed C sampling
        indices = np.linspace(
            0,
            len(X_trimmed) - 1,
            C
        ).astype(int)

        X_fixed = X_trimmed[indices]

        # flatten again
        row = X_fixed.flatten().tolist()
        row.append(target)

        # column names
        columns = []

        joints = [
            "head",
            "left_shoulder", "left_elbow",
            "right_shoulder", "right_elbow",
            "left_hand", "right_hand",
            "left_hip", "right_hip",
            "left_knee", "right_knee",
            "left_foot", "right_foot"
        ]

        for frame_idx in range(C):
            for joint in joints:
                columns += [
                    f"frame{frame_idx}_{joint}_x",
                    f"frame{frame_idx}_{joint}_y",
                    f"frame{frame_idx}_{joint}_z"
                ]

        columns.append("target")

        output_df = pd.DataFrame([row], columns=columns)

        output_path = os.path.join(
            output_folder,
            file_name.replace("_padded", "_fixedC")
        )

        output_df.to_csv(output_path, index=False)

        print(f"Saved: {output_path}")

    print("\nDone.")

In [7]:
convert_padded_csv_to_fixed_c(
    input_folder="../../MainProject/data/mediapipe_padded_videos",
    output_folder="../../MainProject/data/mediapipe_fixed_c_bad_good_videos",
    C=30
)


Processing: G22_padded.csv
Original frames: 173
After zero removal: 66
Saved: ../../MainProject/data/mediapipe_fixed_c_bad_good_videos/G22_fixedC.csv

Processing: G40_padded.csv
Original frames: 173
After zero removal: 66
Saved: ../../MainProject/data/mediapipe_fixed_c_bad_good_videos/G40_fixedC.csv

Processing: G77_padded.csv
Original frames: 173
After zero removal: 63
Saved: ../../MainProject/data/mediapipe_fixed_c_bad_good_videos/G77_fixedC.csv

Processing: G09_padded.csv
Original frames: 173
After zero removal: 79
Saved: ../../MainProject/data/mediapipe_fixed_c_bad_good_videos/G09_fixedC.csv

Processing: G66_padded.csv
Original frames: 173
After zero removal: 74
Saved: ../../MainProject/data/mediapipe_fixed_c_bad_good_videos/G66_fixedC.csv

Processing: G51_padded.csv
Original frames: 173
After zero removal: 76
Saved: ../../MainProject/data/mediapipe_fixed_c_bad_good_videos/G51_fixedC.csv

Processing: G04_padded.csv
Original frames: 173
After zero removal: 65
Saved: ../../MainProje

In [10]:
df = pd.read_csv("../../MainProject/data/mediapipe_fixed_c_bad_good_videos/G01_fixedC.csv")

y = df["target"].values

X_flat = df.drop(columns=["target"]).values

max_frames = 30 
n_features = 39

X = X_flat.reshape(-1, max_frames, n_features)

print(X.shape)

print(X)
print(y)

(1, 30, 39)
[[[ 0.52720726  0.16865908 -0.32241583 ...  0.49026176  0.78741807
    0.16503   ]
  [ 0.52691102  0.1725561  -0.35200885 ...  0.49016017  0.79354775
    0.16988555]
  [ 0.52686512  0.17563368 -0.37094072 ...  0.49094567  0.79339057
    0.15731364]
  ...
  [ 0.52449417  0.25359911 -0.34633252 ...  0.49011758  0.79879624
   -0.02932534]
  [ 0.52444386  0.21728952 -0.36574191 ...  0.49059886  0.79905272
    0.09119117]
  [ 0.52430266  0.17183761 -0.28378439 ...  0.49128568  0.79606318
    0.14232932]]]
[1]
